### best model for rf

# 1- DATA PREPARATION:
1-Loading the composition dataset.
2-Creating the group_id variable based on gas composition features.
3-Performing an 80/20 GroupShuffleSplit for training and testing datasets.
4-Verifying the absence of data leakage by ensuring no group overlap exists between the training and testing sets.
5-Creating a 5-fold GroupKFold cross-validation scheme, which will later be used to generate out-of-fold (OOF) predictions for the Random Forest (RF), XGBoost (XGB), and Artificial Neural Network (ANN) base learners within the stacking framework.

In [43]:
# ==========================================================
# STAGE 1: DATA PREPARATION
# ==========================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit, GroupKFold

# ==========================================================
# LOAD DATA
# ==========================================================
data = pd.read_csv("output.csv")
data.columns = data.columns.str.strip()

# ==========================================================
# FEATURES & TARGET
# ==========================================================
target_col = "Hydrate Temperature (K)"

feature_cols = [
    col for col in data.columns
    if col != target_col
]

# ==========================================================
# GROUPING
# Remove Pressure from grouping to ensure
# same gas composition cannot appear in both
# train and test sets at different pressures
# ==========================================================
group_cols = [
    col for col in feature_cols
    if "Pressure" not in col
]

data["group_id"] = (
    data[group_cols]
    .round(3)
    .astype(str)
    .agg("_".join, axis=1)
)

# ==========================================================
# INPUTS
# ==========================================================
X = data[feature_cols].values
y = data[target_col].values
groups = data["group_id"]

print("Dataset Shape:", X.shape)

# ==========================================================
# TRAIN / TEST SPLIT
# ==========================================================
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

X_train = X[train_idx]
X_test  = X[test_idx]

y_train = y[train_idx]
y_test  = y[test_idx]

groups_train = groups.iloc[train_idx]

# ==========================================================
# LEAKAGE CHECK
# ==========================================================
overlap = len(
    set(groups_train) &
    set(groups.iloc[test_idx])
)

print("Leakage Check =", overlap)

# ==========================================================
# GROUP KFOLD FOR OOF STACKING
# ==========================================================
gkf = GroupKFold(n_splits=5)

cv_splits = list(
    gkf.split(
        X_train,
        y_train,
        groups_train
    )
)

print("Number of folds:", len(cv_splits))

# ==========================================================
# SUMMARY
# ==========================================================
print("\nTrain Size:", len(X_train))
print("Test Size :", len(X_test))
print("Unique Train Groups:", groups_train.nunique())
print("Unique Test Groups :", groups.iloc[test_idx].nunique())

Dataset Shape: (1036, 12)
Leakage Check = 0
Number of folds: 5

Train Size: 824
Test Size : 212
Unique Train Groups: 12
Unique Test Groups : 3


# 2- Random Forest Out-of-Fold Prediction Generation
1- The optimized Random Forest model was trained using the predefined GroupKFold cross-validation splits.
2- For each fold, predictions were generated for the unseen validation subset and stored as Out-of-Fold (OOF) predictions.
3- The fold-specific Random Forest model was also used to generate predictions for the independent test set.
4- Test-set predictions obtained from all folds were averaged to produce a robust final Random Forest prediction.
5- The resulting OOF training predictions and averaged test predictions were saved for subsequent use as meta-features in the stacking framework.

In [44]:
# ==========================================================
# RF GRID SEARCH + OOF GENERATION + SAVE MODEL
# ==========================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import (
    GridSearchCV,
    KFold
)

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# ==========================================================
# METRICS
# ==========================================================

def rmse(y_true, y_pred):
    return np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

def rrmse(y_true, y_pred):
    return (
        rmse(y_true, y_pred)
        /
        np.mean(y_true)
    ) * 100

def mape(y_true, y_pred):
    return np.mean(
        np.abs(
            (y_true - y_pred)
            /
            y_true
        )
    ) * 100

# ==========================================================
# KFOLD
# ==========================================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_splits = list(
    kf.split(X_train)
)

# ==========================================================
# GRID SEARCH
# ==========================================================

rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

param_grid = {

    "n_estimators": [
        200,
        300
    ],

    "max_depth": [
        
        12,
        14,
        
    ],

    "min_samples_split": [
        2
    ],

    "min_samples_leaf": [
        1,
        2
    ]
}

grid_rf = GridSearchCV(

    estimator=rf,

    param_grid=param_grid,

    scoring="r2",

    cv=kf,

    n_jobs=-1,

    verbose=2
)

# ==========================================================
# FIT GRID SEARCH
# ==========================================================

grid_rf.fit(
    X_train,
    y_train
)

best_rf = grid_rf.best_estimator_

print("\nBest Parameters")
print(grid_rf.best_params_)

print("\nBest CV R2")
print(grid_rf.best_score_)

# ==========================================================
# OOF ARRAYS
# ==========================================================

rf_oof_train = np.zeros(
    len(X_train)
)

rf_test_folds = np.zeros(
    (
        len(X_test),
        len(cv_splits)
    )
)

# ==========================================================
# OOF LOOP
# ==========================================================

for fold, (tr_idx, val_idx) in enumerate(
    cv_splits
):

    print(
        f"\nRF Fold {fold+1}"
    )

    X_tr = X_train[tr_idx]
    y_tr = y_train[tr_idx]

    X_val = X_train[val_idx]

    rf_fold = RandomForestRegressor(

        **grid_rf.best_params_,

        random_state=42,
        n_jobs=-1
    )

    rf_fold.fit(
        X_tr,
        y_tr
    )

    rf_oof_train[val_idx] = (
        rf_fold.predict(X_val)
    )

    rf_test_folds[:, fold] = (
        rf_fold.predict(X_test)
    )

# ==========================================================
# FINAL TEST PREDICTION
# ==========================================================

rf_test_pred = rf_test_folds.mean(
    axis=1
)

# ==========================================================
# OOF METRICS
# ==========================================================

oof_r2 = r2_score(
    y_train,
    rf_oof_train
)

oof_rmse = rmse(
    y_train,
    rf_oof_train
)

oof_mae = mean_absolute_error(
    y_train,
    rf_oof_train
)

oof_rrmse = rrmse(
    y_train,
    rf_oof_train
)

oof_mape = mape(
    y_train,
    rf_oof_train
)

# ==========================================================
# TEST METRICS
# ==========================================================

test_r2 = r2_score(
    y_test,
    rf_test_pred
)

test_rmse = rmse(
    y_test,
    rf_test_pred
)

test_mae = mean_absolute_error(
    y_test,
    rf_test_pred
)

test_rrmse = rrmse(
    y_test,
    rf_test_pred
)

test_mape = mape(
    y_test,
    rf_test_pred
)

# ==========================================================
# DISPLAY RESULTS
# ==========================================================

print("\n===== RF OOF RESULTS =====")

print(f"R2       : {oof_r2:.6f}")
print(f"RMSE     : {oof_rmse:.6f}")
print(f"MAE      : {oof_mae:.6f}")
print(f"RRMSE %  : {oof_rrmse:.6f}")
print(f"MAPE %   : {oof_mape:.6f}")

print("\n===== RF TEST RESULTS =====")

print(f"R2       : {test_r2:.6f}")
print(f"RMSE     : {test_rmse:.6f}")
print(f"MAE      : {test_mae:.6f}")
print(f"RRMSE %  : {test_rrmse:.6f}")
print(f"MAPE %   : {test_mape:.6f}")

# ==========================================================
# TRAIN FINAL RF ON FULL TRAINING SET
# ==========================================================

final_rf = RandomForestRegressor(

    **grid_rf.best_params_,

    random_state=42,
    n_jobs=-1
)

final_rf.fit(
    X_train,
    y_train
)

# ==========================================================
# SAVE FINAL MODEL
# ==========================================================

joblib.dump(
    final_rf,
    "RF_BaseLearner1.pkl"
)

# ==========================================================
# SAVE OOF FILES
# ==========================================================

joblib.dump(
    rf_oof_train,
    "rf_oof_train.pkl"
)

joblib.dump(
    rf_test_pred,
    "rf_test_pred.pkl"
)

# ==========================================================
# SAVE METRICS
# ==========================================================

pd.DataFrame([{

    "OOF_R2": oof_r2,
    "OOF_RMSE": oof_rmse,
    "OOF_MAE": oof_mae,
    "OOF_RRMSE": oof_rrmse,
    "OOF_MAPE": oof_mape,

    "TEST_R2": test_r2,
    "TEST_RMSE": test_rmse,
    "TEST_MAE": test_mae,
    "TEST_RRMSE": test_rrmse,
    "TEST_MAPE": test_mape

}]).to_csv(
    "RF_OOF_Test_Metrics.csv",
    index=False
)

print("\nRF Base Learner Saved")
print("OOF Features Saved")
print("Metrics Saved")

Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best Parameters
{'max_depth': 14, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}

Best CV R2
0.9967981887189203

RF Fold 1

RF Fold 2

RF Fold 3

RF Fold 4

RF Fold 5

===== RF OOF RESULTS =====
R2       : 0.996627
RMSE     : 0.900471
MAE      : 0.452010
RRMSE %  : 0.319344
MAPE %   : 0.171617

===== RF TEST RESULTS =====
R2       : 0.998547
RMSE     : 0.605373
MAE      : 0.368540
RRMSE %  : 0.215710
MAPE %   : 0.137812

RF Base Learner Saved
OOF Features Saved
Metrics Saved


# 3- XGBoost Out-of-Fold Prediction Generation 
1- The optimized XGBoost hyperparameters obtained from the previous GroupKFold grid search were fixed for OOF generation.
2- The training dataset was divided using the predefined five-fold GroupKFold strategy.
3- An XGBoost model was trained on each training fold and used to predict the corresponding unseen validation fold.
4- The validation predictions from all folds were combined to create leakage-free Out-of-Fold (OOF) predictions.
5- Predictions for the independent test set were generated in every fold and averaged to obtain the final XGBoost meta-features.

In [45]:
# ==========================================================
# STAGE 3: XGBOOST GRID SEARCH + OOF GENERATION
# + TEST EVALUATION + SAVE MODEL
# ==========================================================

import numpy as np
import pandas as pd
import joblib

from xgboost import XGBRegressor

from sklearn.model_selection import (
    GridSearchCV,
    KFold
)

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# ==========================================================
# METRICS
# ==========================================================

def rmse(y_true, y_pred):
    return np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

def rrmse(y_true, y_pred):
    return (
        rmse(y_true, y_pred)
        /
        np.mean(y_true)
    ) * 100

def mape(y_true, y_pred):
    return np.mean(
        np.abs(
            (y_true - y_pred)
            /
            y_true
        )
    ) * 100

# ==========================================================
# KFOLD
# ==========================================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_splits = list(
    kf.split(X_train)
)

# ==========================================================
# XGB GRID SEARCH
# ==========================================================

xgb = XGBRegressor(

    objective="reg:squarederror",

    random_state=42,

    n_jobs=-1
)

param_grid = {

    "n_estimators": [
        200,
        300,
        500
    ],

    "max_depth": [
        3,
        4,
       
    ],

    "learning_rate": [
        0.01,
        0.03
    ],

    "subsample": [
        0.8
    ],

    "colsample_bytree": [
        0.8,
        0.9
    ]
}

grid_xgb = GridSearchCV(

    estimator=xgb,

    param_grid=param_grid,

    scoring="r2",

    cv=kf,

    n_jobs=-1,

    verbose=2,

    return_train_score=True
)

# ==========================================================
# FIT GRID SEARCH
# ==========================================================

grid_xgb.fit(
    X_train,
    y_train
)

best_params = grid_xgb.best_params_

print("\n" + "="*60)
print("BEST XGB PARAMETERS")
print("="*60)

print(best_params)

print("\nBEST CV R²")
print(grid_xgb.best_score_)

# ==========================================================
# OOF ARRAYS
# ==========================================================

xgb_oof_train = np.zeros(
    len(X_train)
)

xgb_test_folds = np.zeros(
    (
        len(X_test),
        len(cv_splits)
    )
)

fold_metrics = []

# ==========================================================
# OOF LOOP
# ==========================================================

for fold, (tr_idx, val_idx) in enumerate(
    cv_splits
):

    print(
        f"\nXGB Fold {fold+1}"
    )

    X_tr = X_train[tr_idx]
    y_tr = y_train[tr_idx]

    X_val = X_train[val_idx]
    y_val = y_train[val_idx]

    xgb_fold = XGBRegressor(

        **best_params,

        objective="reg:squarederror",

        random_state=42,

        n_jobs=-1
    )

    xgb_fold.fit(
        X_tr,
        y_tr
    )

    # ======================================================
    # OOF Prediction
    # ======================================================

    y_val_pred = xgb_fold.predict(
        X_val
    )

    xgb_oof_train[val_idx] = (
        y_val_pred
    )

    # ======================================================
    # Test Prediction
    # ======================================================

    xgb_test_folds[:, fold] = (
        xgb_fold.predict(X_test)
    )

    # ======================================================
    # Fold Metrics
    # ======================================================

    fold_metrics.append({

        "Fold":
            fold + 1,

        "R2":
            r2_score(
                y_val,
                y_val_pred
            ),

        "RMSE":
            rmse(
                y_val,
                y_val_pred
            ),

        "MAE":
            mean_absolute_error(
                y_val,
                y_val_pred
            ),

        "RRMSE":
            rrmse(
                y_val,
                y_val_pred
            ),

        "MAPE":
            mape(
                y_val,
                y_val_pred
            )
    })

# ==========================================================
# FINAL TEST PREDICTION
# ==========================================================

xgb_test_pred = (
    xgb_test_folds.mean(axis=1)
)

# ==========================================================
# OOF METRICS
# ==========================================================

oof_r2 = r2_score(
    y_train,
    xgb_oof_train
)

oof_rmse = rmse(
    y_train,
    xgb_oof_train
)

oof_mae = mean_absolute_error(
    y_train,
    xgb_oof_train
)

oof_rrmse = rrmse(
    y_train,
    xgb_oof_train
)

oof_mape = mape(
    y_train,
    xgb_oof_train
)

# ==========================================================
# TEST METRICS
# ==========================================================

test_r2 = r2_score(
    y_test,
    xgb_test_pred
)

test_rmse = rmse(
    y_test,
    xgb_test_pred
)

test_mae = mean_absolute_error(
    y_test,
    xgb_test_pred
)

test_rrmse = rrmse(
    y_test,
    xgb_test_pred
)

test_mape = mape(
    y_test,
    xgb_test_pred
)

# ==========================================================
# DISPLAY RESULTS
# ==========================================================

print("\n===== XGB OOF RESULTS =====")

print(f"R2       : {oof_r2:.6f}")
print(f"RMSE     : {oof_rmse:.6f}")
print(f"MAE      : {oof_mae:.6f}")
print(f"RRMSE %  : {oof_rrmse:.6f}")
print(f"MAPE %   : {oof_mape:.6f}")

print("\n===== XGB TEST RESULTS =====")

print(f"R2       : {test_r2:.6f}")
print(f"RMSE     : {test_rmse:.6f}")
print(f"MAE      : {test_mae:.6f}")
print(f"RRMSE %  : {test_rrmse:.6f}")
print(f"MAPE %   : {test_mape:.6f}")

# ==========================================================
# TRAIN FINAL MODEL
# ==========================================================

final_xgb = XGBRegressor(

    **best_params,

    objective="reg:squarederror",

    random_state=42,

    n_jobs=-1
)

final_xgb.fit(
    X_train,
    y_train
)

# ==========================================================
# SAVE FINAL MODEL
# ==========================================================

joblib.dump(
    final_xgb,
    "XGB_BaseLearner1.pkl"
)

# ==========================================================
# SAVE OOF FEATURES
# ==========================================================

joblib.dump(
    xgb_oof_train,
    "xgb_oof_train.pkl"
)

joblib.dump(
    xgb_test_pred,
    "xgb_test_pred.pkl"
)

# ==========================================================
# SAVE FOLD METRICS
# ==========================================================

pd.DataFrame(
    fold_metrics
).to_csv(
    "XGB_Fold_Metrics.csv",
    index=False
)

# ==========================================================
# SAVE SUMMARY
# ==========================================================

summary = pd.DataFrame([{

    "Best_CV_R2":
        grid_xgb.best_score_,

    "OOF_R2":
        oof_r2,

    "OOF_RMSE":
        oof_rmse,

    "OOF_MAE":
        oof_mae,

    "OOF_RRMSE":
        oof_rrmse,

    "OOF_MAPE":
        oof_mape,

    "TEST_R2":
        test_r2,

    "TEST_RMSE":
        test_rmse,

    "TEST_MAE":
        test_mae,

    "TEST_RRMSE":
        test_rrmse,

    "TEST_MAPE":
        test_mape

}])

summary.to_csv(
    "XGB_OOF_Test_Summary.csv",
    index=False
)

# ==========================================================
# SAVE BEST PARAMETERS
# ==========================================================

pd.DataFrame(
    [best_params]
).to_csv(
    "XGB_Best_Params.csv",
    index=False
)

print("\n" + "="*60)
print("XGB GRID SEARCH COMPLETED")
print("XGB_BaseLearner.pkl SAVED")
print("OOF FEATURES SAVED")
print("METRICS SAVED")
print("="*60)

Fitting 5 folds for each of 24 candidates, totalling 120 fits

BEST XGB PARAMETERS
{'colsample_bytree': 0.9, 'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 500, 'subsample': 0.8}

BEST CV R²
0.9986864676494495

XGB Fold 1

XGB Fold 2

XGB Fold 3

XGB Fold 4

XGB Fold 5

===== XGB OOF RESULTS =====
R2       : 0.998667
RMSE     : 0.566055
MAE      : 0.320776
RRMSE %  : 0.200747
MAPE %   : 0.119928

===== XGB TEST RESULTS =====
R2       : 0.997416
RMSE     : 0.807262
MAE      : 0.531789
RRMSE %  : 0.287648
MAPE %   : 0.196279

XGB GRID SEARCH COMPLETED
XGB_BaseLearner.pkl SAVED
OOF FEATURES SAVED
METRICS SAVED


# 4- ANN Out-of-Fold Prediction Generation
1- The ANN model was retrained within each GroupKFold split using only the training portion of the fold.
2- Feature and target scaling were fitted exclusively on the fold training data to prevent information leakage.
3- Predictions were generated for the unseen validation fold and stored as Out-of-Fold (OOF) predictions.
4- Each fold-specific ANN model was also used to generate predictions for the independent test set.
5- The test predictions from all folds were averaged and saved together with the OOF predictions for subsequent stacking.

In [46]:
# ==========================================================
# STAGE 4: ANN OOF GENERATION + FINAL MODEL SAVE
# ==========================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, ELU
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# ==========================================================
# METRICS
# ==========================================================

def rmse(y_true, y_pred):
    return np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

def rrmse(y_true, y_pred):
    return (
        rmse(y_true, y_pred)
        /
        np.mean(y_true)
    )

def mape(y_true, y_pred):
    return np.mean(
        np.abs(
            (y_true - y_pred)
            /
            y_true
        )
    ) * 100

# ==========================================================
# OOF ARRAYS
# ==========================================================

ann_oof_train = np.zeros(
    len(X_train)
)

ann_test_folds = np.zeros(
    (
        len(X_test),
        len(cv_splits)
    )
)

fold_metrics = []

# ==========================================================
# GROUPKFOLD LOOP
# ==========================================================

for fold, (tr_idx, val_idx) in enumerate(cv_splits):

    print(f"\nANN Fold {fold+1}")

    X_tr = X_train[tr_idx]
    y_tr = y_train[tr_idx]

    X_val = X_train[val_idx]
    y_val = y_train[val_idx]

    # ======================================================
    # SCALING
    # ======================================================

    X_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_tr_scaled = X_scaler.fit_transform(
        X_tr
    )

    X_val_scaled = X_scaler.transform(
        X_val
    )

    X_test_scaled = X_scaler.transform(
        X_test
    )

    y_tr_scaled = y_scaler.fit_transform(
        y_tr.reshape(-1, 1)
    ).ravel()

    # ======================================================
    # BUILD MODEL
    # ======================================================

    model = Sequential()

    model.add(
        Dense(
            256,
            input_dim=X_tr.shape[1]
        )
    )
    model.add(ELU())

    model.add(Dense(128))
    model.add(ELU())

    model.add(Dense(64))
    model.add(ELU())

    model.add(Dense(1))

    model.compile(
        optimizer=Adam(
            learning_rate=0.001
        ),
        loss="mse"
    )

    # ======================================================
    # EARLY STOPPING
    # ======================================================

    early = EarlyStopping(
        monitor="val_loss",
        patience=40,
        restore_best_weights=True
    )

    # ======================================================
    # TRAIN
    # ======================================================

    model.fit(
        X_tr_scaled,
        y_tr_scaled,
        validation_split=0.1,
        epochs=1500,
        batch_size=16,
        callbacks=[early],
        verbose=0
    )

    # ======================================================
    # VALIDATION PREDICTION
    # ======================================================

    y_val_pred_scaled = model.predict(
        X_val_scaled,
        verbose=0
    )

    y_val_pred = y_scaler.inverse_transform(
        y_val_pred_scaled
    ).ravel()

    ann_oof_train[val_idx] = y_val_pred

    # ======================================================
    # TEST PREDICTION
    # ======================================================

    y_test_pred_scaled = model.predict(
        X_test_scaled,
        verbose=0
    )

    y_test_pred = y_scaler.inverse_transform(
        y_test_pred_scaled
    ).ravel()

    ann_test_folds[:, fold] = y_test_pred

    # ======================================================
    # FOLD METRICS
    # ======================================================

    fold_metrics.append({

        "Fold": fold + 1,

        "R2":
            r2_score(
                y_val,
                y_val_pred
            ),

        "RMSE":
            rmse(
                y_val,
                y_val_pred
            ),

        "MAE":
            mean_absolute_error(
                y_val,
                y_val_pred
            ),

        "RRMSE":
            rrmse(
                y_val,
                y_val_pred
            ),

        "MAPE":
            mape(
                y_val,
                y_val_pred
            )
    })

# ==========================================================
# FINAL TEST PREDICTION
# ==========================================================

ann_test_pred = ann_test_folds.mean(
    axis=1
)

# ==========================================================
# OOF METRICS
# ==========================================================

ann_r2 = r2_score(
    y_train,
    ann_oof_train
)

ann_rmse = rmse(
    y_train,
    ann_oof_train
)

ann_mae = mean_absolute_error(
    y_train,
    ann_oof_train
)

ann_rrmse = rrmse(
    y_train,
    ann_oof_train
)

ann_mape = mape(
    y_train,
    ann_oof_train
)

# ==========================================================
# TEST METRICS
# ==========================================================

test_r2 = r2_score(
    y_test,
    ann_test_pred
)

test_rmse = rmse(
    y_test,
    ann_test_pred
)

test_mae = mean_absolute_error(
    y_test,
    ann_test_pred
)

test_rrmse = rrmse(
    y_test,
    ann_test_pred
)

test_mape = mape(
    y_test,
    ann_test_pred
)

# ==========================================================
# DISPLAY RESULTS
# ==========================================================

print("\n===== ANN OOF RESULTS =====")

print(f"R2       : {ann_r2:.6f}")
print(f"RMSE     : {ann_rmse:.6f}")
print(f"MAE      : {ann_mae:.6f}")
print(f"RRMSE    : {ann_rrmse:.6f}")
print(f"MAPE (%) : {ann_mape:.6f}")

print("\n===== ANN TEST RESULTS =====")

print(f"R2       : {test_r2:.6f}")
print(f"RMSE     : {test_rmse:.6f}")
print(f"MAE      : {test_mae:.6f}")
print(f"RRMSE    : {test_rrmse:.6f}")
print(f"MAPE (%) : {test_mape:.6f}")

# ==========================================================
# SAVE OOF FEATURES
# ==========================================================

joblib.dump(
    ann_oof_train,
    "ann_oof_train.pkl"
)

joblib.dump(
    ann_test_pred,
    "ann_test_pred.pkl"
)

# ==========================================================
# SAVE METRICS
# ==========================================================

pd.DataFrame(
    fold_metrics
).to_csv(
    "ann_fold_metrics.csv",
    index=False
)

pd.DataFrame([{

    "R2": ann_r2,
    "RMSE": ann_rmse,
    "MAE": ann_mae,
    "RRMSE": ann_rrmse,
    "MAPE": ann_mape

}]).to_csv(
    "ann_oof_metrics.csv",
    index=False
)

pd.DataFrame([{

    "R2": test_r2,
    "RMSE": test_rmse,
    "MAE": test_mae,
    "RRMSE": test_rrmse,
    "MAPE": test_mape

}]).to_csv(
    "ann_test_metrics.csv",
    index=False
)

# ==========================================================
# SAVE PREDICTIONS
# ==========================================================

pd.DataFrame({

    "Actual": y_train,
    "OOF_Prediction": ann_oof_train

}).to_csv(
    "ann_oof_predictions.csv",
    index=False
)

pd.DataFrame({

    "Actual": y_test,
    "Test_Prediction": ann_test_pred

}).to_csv(
    "ann_test_predictions.csv",
    index=False
)

# ==========================================================
# TRAIN FINAL ANN ON FULL TRAIN
# ==========================================================

print(
    "\nTraining Final ANN On Full Training Set..."
)

X_scaler_final = StandardScaler()
y_scaler_final = StandardScaler()

X_train_scaled = X_scaler_final.fit_transform(
    X_train
)

y_train_scaled = y_scaler_final.fit_transform(
    y_train.reshape(-1, 1)
).ravel()

final_ann = Sequential()

final_ann.add(
    Dense(
        256,
        input_dim=X_train.shape[1]
    )
)
final_ann.add(ELU())

final_ann.add(Dense(128))
final_ann.add(ELU())

final_ann.add(Dense(64))
final_ann.add(ELU())

final_ann.add(Dense(1))

final_ann.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),
    loss="mse"
)

early = EarlyStopping(
    monitor="val_loss",
    patience=40,
    restore_best_weights=True
)

final_ann.fit(
    X_train_scaled,
    y_train_scaled,
    validation_split=0.1,
    epochs=1500,
    batch_size=16,
    callbacks=[early],
    verbose=1
)

# ==========================================================
# SAVE FINAL MODEL
# ==========================================================

final_ann.save(
    "ann_base_final.keras"
)

joblib.dump(
    X_scaler_final,
    "ann_X_scaler.pkl"
)

joblib.dump(
    y_scaler_final,
    "ann_y_scaler.pkl"
)

print("\nANN OOF Generation Completed")
print("ANN Final Model Saved")

print("ann_base_final.keras")
print("ann_X_scaler.pkl")
print("ann_y_scaler.pkl")


ANN Fold 1

ANN Fold 2

ANN Fold 3

ANN Fold 4

ANN Fold 5

===== ANN OOF RESULTS =====
R2       : 0.967285
RMSE     : 2.804541
MAE      : 1.473619
RRMSE    : 0.009946
MAPE (%) : 0.551996

===== ANN TEST RESULTS =====
R2       : 0.991404
RMSE     : 1.472364
MAE      : 0.898256
RRMSE    : 0.005246
MAPE (%) : 0.339196

Training Final ANN On Full Training Set...
Epoch 1/1500
47/47 [==============================] - 2s 9ms/step - loss: 0.4296 - val_loss: 0.5744
Epoch 2/1500
47/47 [==============================] - 0s 5ms/step - loss: 0.3256 - val_loss: 0.5169
Epoch 3/1500
47/47 [==============================] - 0s 5ms/step - loss: 0.2684 - val_loss: 0.6127
Epoch 4/1500
47/47 [==============================] - 0s 5ms/step - loss: 0.3059 - val_loss: 0.5930
Epoch 5/1500
47/47 [==============================] - 0s 5ms/step - loss: 0.2650 - val_loss: 0.4596
Epoch 6/1500
47/47 [==============================] - 0s 5ms/step - loss: 0.2462 - val_loss: 0.5213
Epoch 7/1500
47/47 [=================

# 5- Meta Dataset Construction
1- Loaded the OOF predictions generated by RF, XGBoost, and ANN.
2- Loaded the averaged test-set predictions produced by the three base learners.
3- Constructed the meta-training dataset using RF, XGBoost, and ANN OOF predictions as input features.
4- Constructed the meta-testing dataset using the corresponding test predictions from the three base learners.
5- Assigned the original hydrate equilibrium temperatures (y_train and y_test) as the target values for meta-learner training and evaluation.

In [47]:
# ==========================================================
# STAGE 5: META DATASET CONSTRUCTION
# ==========================================================

import numpy as np
import pandas as pd
import joblib

# ==========================================================
# LOAD OOF PREDICTIONS
# ==========================================================
rf_oof_train = joblib.load("rf_oof_train.pkl")
xgb_oof_train = joblib.load("xgb_oof_train.pkl")
ann_oof_train = joblib.load("ann_oof_train.pkl")

# ==========================================================
# LOAD TEST PREDICTIONS
# ==========================================================
rf_test_pred = joblib.load("rf_test_pred.pkl")
xgb_test_pred = joblib.load("xgb_test_pred.pkl")
ann_test_pred = joblib.load("ann_test_pred.pkl")

# ==========================================================
# META TRAIN DATASET
# ==========================================================
X_meta_train = np.column_stack([
    rf_oof_train,
    xgb_oof_train,
    ann_oof_train
])

# ==========================================================
# META TEST DATASET
# ==========================================================
X_meta_test = np.column_stack([
    rf_test_pred,
    xgb_test_pred,
    ann_test_pred
])

# ==========================================================
# TARGET
# ==========================================================
y_meta_train = y_train
y_meta_test = y_test

# ==========================================================
# SAVE META DATASETS
# ==========================================================
joblib.dump(X_meta_train, "X_meta_train.pkl")
joblib.dump(X_meta_test, "X_meta_test.pkl")

joblib.dump(y_meta_train, "y_meta_train.pkl")
joblib.dump(y_meta_test, "y_meta_test.pkl")

# Optional CSV Files
pd.DataFrame(
    X_meta_train,
    columns=["RF", "XGB", "ANN"]
).to_csv("meta_train_dataset.csv", index=False)

pd.DataFrame(
    X_meta_test,
    columns=["RF", "XGB", "ANN"]
).to_csv("meta_test_dataset.csv", index=False)

print("Meta Dataset Construction Completed")
print("X_meta_train shape:", X_meta_train.shape)
print("X_meta_test shape:", X_meta_test.shape)

Meta Dataset Construction Completed
X_meta_train shape: (824, 3)
X_meta_test shape: (212, 3)


# 6- Meta Learner (XGBoost)
1- Loaded the meta-training and meta-testing datasets generated from RF, XGBoost, and ANN predictions.
2- Defined XGBoost as the meta learner responsible for combining the outputs of the three base learners.
3- Performed hyperparameter optimization using GridSearchCV.
4- Trained the optimal XGBoost meta model on the complete meta-training dataset.
5- Evaluated the stacked model on the independent test set and saved the final model and predictions.

In [48]:
# ==========================================================
# STAGE 6: META LEARNER (XGBOOST)
# ==========================================================

import numpy as np
import pandas as pd
import joblib

from xgboost import XGBRegressor

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# ==========================================================
# LOAD META DATA
# ==========================================================

X_meta_train = joblib.load("X_meta_train.pkl")
X_meta_test  = joblib.load("X_meta_test.pkl")

y_meta_train = joblib.load("y_meta_train.pkl")
y_meta_test  = joblib.load("y_meta_test.pkl")

print("Train Shape:", X_meta_train.shape)
print("Test Shape :", X_meta_test.shape)

# ==========================================================
# METRICS
# ==========================================================

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def rrmse(y_true, y_pred):
    return rmse(y_true, y_pred) / np.mean(y_true)

def mape(y_true, y_pred):
    return np.mean(
        np.abs((y_true - y_pred) / y_true)
    ) * 100

# ==========================================================
# META XGB MODEL
# ==========================================================

meta_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# ==========================================================
# HYPERPARAMETER GRID
# ==========================================================

param_grid = {

    "n_estimators": [100, 200, 300],

    "learning_rate": [0.03, 0.05, 0.1],

    "max_depth": [2, 3, 4],

    "subsample": [0.8, 1.0],

    "colsample_bytree": [0.8, 1.0]
}

# ==========================================================
# GRID SEARCH
# ==========================================================

grid = GridSearchCV(
    estimator=meta_xgb,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid.fit(
    X_meta_train,
    y_meta_train
)

# ==========================================================
# BEST MODEL
# ==========================================================

best_meta_xgb = grid.best_estimator_

print("\nBest Parameters:")
print(grid.best_params_)

print("\nBest CV R2:")
print(grid.best_score_)

# ==========================================================
# TEST PREDICTION
# ==========================================================

y_pred = best_meta_xgb.predict(
    X_meta_test
)

# ==========================================================
# TEST METRICS
# ==========================================================

test_r2 = r2_score(
    y_meta_test,
    y_pred
)

test_rmse = rmse(
    y_meta_test,
    y_pred
)

test_mae = mean_absolute_error(
    y_meta_test,
    y_pred
)

test_rrmse = rrmse(
    y_meta_test,
    y_pred
)

test_mape = mape(
    y_meta_test,
    y_pred
)

# ==========================================================
# RESULTS
# ==========================================================

print("\n===== META XGB RESULTS =====")

print(f"CV R2    : {grid.best_score_:.6f}")
print(f"Test R2  : {test_r2:.6f}")
print(f"RMSE     : {test_rmse:.6f}")
print(f"MAE      : {test_mae:.6f}")
print(f"RRMSE    : {test_rrmse:.6f}")
print(f"MAPE (%) : {test_mape:.6f}")

# ==========================================================
# SAVE MODEL
# ==========================================================

joblib.dump(
    best_meta_xgb,
    "XGB_meta_model.pkl"
)

# ==========================================================
# SAVE PREDICTIONS
# ==========================================================

pred_df = pd.DataFrame({
    "Actual": y_meta_test,
    "Predicted": y_pred
})

pred_df.to_csv(
    "XGB_meta_predictions.csv",
    index=False
)

# ==========================================================
# SAVE RESULTS
# ==========================================================

results_df = pd.DataFrame([{
    "CV_R2": grid.best_score_,
    "Test_R2": test_r2,
    "RMSE": test_rmse,
    "MAE": test_mae,
    "RRMSE": test_rrmse,
    "MAPE": test_mape
}])

results_df.to_csv(
    "XGB_meta_results.csv",
    index=False
)

print("\nModel Saved -> XGB_meta_model.pkl")
print("Predictions Saved -> XGB_meta_predictions.csv")
print("Results Saved -> XGB_meta_results.csv")

Train Shape: (824, 3)
Test Shape : (212, 3)
Fitting 5 folds for each of 108 candidates, totalling 540 fits

Best Parameters:
{'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 200, 'subsample': 1.0}

Best CV R2:
0.9973619102225701

===== META XGB RESULTS =====
CV R2    : 0.997362
Test R2  : 0.997891
RMSE     : 0.729346
MAE      : 0.378437
RRMSE    : 0.002599
MAPE (%) : 0.143106

Model Saved -> XGB_meta_model.pkl
Predictions Saved -> XGB_meta_predictions.csv
Results Saved -> XGB_meta_results.csv


# 7- Meta Learner (GPR)


In [49]:
# ==========================================================
# STAGE 7: META LEARNER (GPR)
# ==========================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    Matern,
    WhiteKernel,
    ConstantKernel as C
)

# ==========================================================
# LOAD META DATASET
# ==========================================================
X_meta_train = joblib.load("X_meta_train.pkl")
X_meta_test  = joblib.load("X_meta_test.pkl")

y_meta_train = joblib.load("y_meta_train.pkl")
y_meta_test  = joblib.load("y_meta_test.pkl")

# ==========================================================
# METRICS
# ==========================================================
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def rrmse(y_true, y_pred):
    return rmse(y_true, y_pred) / np.mean(y_true)

def mape(y_true, y_pred):
    return np.mean(
        np.abs((y_true - y_pred) / y_true)
    ) * 100

# ==========================================================
# GPR KERNEL
# ==========================================================
kernel = (
    C(1.0)
    * Matern(length_scale=1.0, nu=1.5)
    + WhiteKernel(noise_level=1e-5)
)

# ==========================================================
# META LEARNER
# ==========================================================
meta_model = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-5,
    normalize_y=True,
    n_restarts_optimizer=5,
    random_state=42
)

# ==========================================================
# TRAIN
# ==========================================================
meta_model.fit(
    X_meta_train,
    y_meta_train
)

# ==========================================================
# PREDICT
# ==========================================================
y_pred = meta_model.predict(X_meta_test)

# ==========================================================
# EVALUATION
# ==========================================================
test_r2 = r2_score(
    y_meta_test,
    y_pred
)

test_rmse = rmse(
    y_meta_test,
    y_pred
)

test_mae = mean_absolute_error(
    y_meta_test,
    y_pred
)

test_rrmse = rrmse(
    y_meta_test,
    y_pred
)

test_mape = mape(
    y_meta_test,
    y_pred
)

# ==========================================================
# SAVE MODEL
# ==========================================================
joblib.dump(
    meta_model,
    "GPR_meta_model.pkl"
)

# ==========================================================
# SAVE PREDICTIONS
# ==========================================================
pd.DataFrame({
    "Actual": y_meta_test,
    "Predicted": y_pred
}).to_csv(
    "GPR_meta_predictions.csv",
    index=False
)

# ==========================================================
# SAVE RESULTS
# ==========================================================
pd.DataFrame([{
    "R2": test_r2,
    "RMSE": test_rmse,
    "MAE": test_mae,
    "RRMSE": test_rrmse,
    "MAPE (%)": test_mape
}]).to_csv(
    "GPR_meta_results.csv",
    index=False
)

# ==========================================================
# PRINT
# ==========================================================
print("\n===== META LEARNER (GPR) RESULTS =====")

print(f"R2       : {test_r2:.6f}")
print(f"RMSE     : {test_rmse:.6f}")
print(f"MAE      : {test_mae:.6f}")
print(f"RRMSE    : {test_rrmse:.6f}")
print(f"MAPE (%) : {test_mape:.6f}")

print("\nModel Saved: GPR_meta_model.pkl")


===== META LEARNER (GPR) RESULTS =====
R2       : 0.998018
RMSE     : 0.707038
MAE      : 0.401888
RRMSE    : 0.002519
MAPE (%) : 0.150553

Model Saved: GPR_meta_model.pkl


# 8- MODEL WITHOUT ANN

In [33]:
# ==========================================================
# STAGE 5: META DATASET CONSTRUCTION
# RF + XGB ONLY
# ==========================================================

import numpy as np
import pandas as pd
import joblib

# ==========================================================
# LOAD OOF PREDICTIONS
# ==========================================================

rf_oof_train = joblib.load(
    "rf_oof_train.pkl"
)

xgb_oof_train = joblib.load(
    "xgb_oof_train.pkl"
)

# ==========================================================
# LOAD TEST PREDICTIONS
# ==========================================================

rf_test_pred = joblib.load(
    "rf_test_pred.pkl"
)

xgb_test_pred = joblib.load(
    "xgb_test_pred.pkl"
)

# ==========================================================
# META TRAIN DATASET
# ==========================================================

X_meta_train = np.column_stack([
    rf_oof_train,
    xgb_oof_train
])

# ==========================================================
# META TEST DATASET
# ==========================================================

X_meta_test = np.column_stack([
    rf_test_pred,
    xgb_test_pred
])

# ==========================================================
# TARGET
# ==========================================================

y_meta_train = y_train
y_meta_test = y_test

# ==========================================================
# SAVE META DATASETS
# ==========================================================

joblib.dump(
    X_meta_train,
    "X_meta_train_RF_XGB.pkl"
)

joblib.dump(
    X_meta_test,
    "X_meta_test_RF_XGB.pkl"
)

joblib.dump(
    y_meta_train,
    "y_meta_train_RF_XGB.pkl"
)

joblib.dump(
    y_meta_test,
    "y_meta_test_RF_XGB.pkl"
)

# ==========================================================
# SAVE CSV FILES
# ==========================================================

pd.DataFrame(
    X_meta_train,
    columns=[
        "RF",
        "XGB"
    ]
).to_csv(
    "meta_train_dataset_RF_XGB.csv",
    index=False
)

pd.DataFrame(
    X_meta_test,
    columns=[
        "RF",
        "XGB"
    ]
).to_csv(
    "meta_test_dataset_RF_XGB.csv",
    index=False
)
# ==========================================================
# DISPLAY
# ==========================================================

print("\nMeta Dataset Construction Completed")

print(
    "X_meta_train shape:",
    X_meta_train.shape
)

print(
    "X_meta_test shape:",
    X_meta_test.shape
)

print(
    "Features Used:",
    ["RF", "XGB"]
)


Meta Dataset Construction Completed
X_meta_train shape: (824, 2)
X_meta_test shape: (212, 2)
Features Used: ['RF', 'XGB']


In [34]:
# ==========================================================
# STAGE 6: META LEARNER (XGBOOST)
# RF + XGB STACKING
# ==========================================================

import numpy as np
import pandas as pd
import joblib

from xgboost import XGBRegressor

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# ==========================================================
# LOAD META DATA
# ==========================================================

X_meta_train = joblib.load(
    "X_meta_train_RF_XGB.pkl"
)

X_meta_test = joblib.load(
    "X_meta_test_RF_XGB.pkl"
)

y_meta_train = joblib.load(
    "y_meta_train_RF_XGB.pkl"
)

y_meta_test = joblib.load(
    "y_meta_test_RF_XGB.pkl"
)

print("Train Shape:", X_meta_train.shape)
print("Test Shape :", X_meta_test.shape)

# ==========================================================
# METRICS
# ==========================================================

def rmse(y_true, y_pred):
    return np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

def rrmse(y_true, y_pred):
    return rmse(
        y_true,
        y_pred
    ) / np.mean(y_true)

def mape(y_true, y_pred):
    return np.mean(
        np.abs(
            (y_true - y_pred) / y_true
        )
    ) * 100

# ==========================================================
# META XGB MODEL
# ==========================================================

meta_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# ==========================================================
# HYPERPARAMETER GRID
# ==========================================================

param_grid = {

    "n_estimators": [
        100,
        200,
        300
    ],

    "learning_rate": [
        0.03,
        0.05,
        0.10
    ],

    "max_depth": [
        2,
        3,
        4
    ],

    "subsample": [
        0.8,
        1.0
    ],

    "colsample_bytree": [
        0.8,
        1.0
    ]
}

# ==========================================================
# GRID SEARCH
# ==========================================================

grid = GridSearchCV(
    estimator=meta_xgb,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid.fit(
    X_meta_train,
    y_meta_train
)

# ==========================================================
# BEST MODEL
# ==========================================================

best_meta_xgb = grid.best_estimator_

print("\nBest Parameters:")
print(grid.best_params_)

print("\nBest CV R2:")
print(grid.best_score_)

# ==========================================================
# TEST PREDICTION
# ==========================================================

y_pred = best_meta_xgb.predict(
    X_meta_test
)

# ==========================================================
# TEST METRICS
# ==========================================================

test_r2 = r2_score(
    y_meta_test,
    y_pred
)

test_rmse = rmse(
    y_meta_test,
    y_pred
)

test_mae = mean_absolute_error(
    y_meta_test,
    y_pred
)

test_rrmse = rrmse(
    y_meta_test,
    y_pred
)

test_mape = mape(
    y_meta_test,
    y_pred
)

# ==========================================================
# RESULTS
# ==========================================================

print("\n===== META XGB (RF + XGB) RESULTS =====")

print(f"CV R2    : {grid.best_score_:.6f}")
print(f"Test R2  : {test_r2:.6f}")
print(f"RMSE     : {test_rmse:.6f}")
print(f"MAE      : {test_mae:.6f}")
print(f"RRMSE    : {test_rrmse:.6f}")
print(f"MAPE (%) : {test_mape:.6f}")

# ==========================================================
# SAVE MODEL
# ==========================================================

joblib.dump(
    best_meta_xgb,
    "XGB_meta_RF_XGB_model.pkl"
)

# ==========================================================
# SAVE PREDICTIONS
# ==========================================================

pred_df = pd.DataFrame({
    "Actual": y_meta_test,
    "Predicted": y_pred
})

pred_df.to_csv(
    "XGB_meta_RF_XGB_predictions.csv",
    index=False
)

# ==========================================================
# SAVE RESULTS
# ==========================================================

results_df = pd.DataFrame([{
    "CV_R2": grid.best_score_,
    "Test_R2": test_r2,
    "RMSE": test_rmse,
    "MAE": test_mae,
    "RRMSE": test_rrmse,
    "MAPE": test_mape
}])

results_df.to_csv(
    "XGB_meta_RF_XGB_results.csv",
    index=False
)

# ==========================================================
# FINISH
# ==========================================================

print(
    "\nModel Saved -> XGB_meta_RF_XGB_model.pkl"
)

print(
    "Predictions Saved -> XGB_meta_RF_XGB_predictions.csv"
)

print(
    "Results Saved -> XGB_meta_RF_XGB_results.csv"
)

Train Shape: (824, 2)
Test Shape : (212, 2)
Fitting 5 folds for each of 108 candidates, totalling 540 fits

Best Parameters:
{'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}

Best CV R2:
0.9973399564536048

===== META XGB (RF + XGB) RESULTS =====
CV R2    : 0.997340
Test R2  : 0.997346
RMSE     : 0.818108
MAE      : 0.433103
RRMSE    : 0.002915
MAPE (%) : 0.162803

Model Saved -> XGB_meta_RF_XGB_model.pkl
Predictions Saved -> XGB_meta_RF_XGB_predictions.csv
Results Saved -> XGB_meta_RF_XGB_results.csv


# 9- MODEL WITHOUT RF

In [35]:
# ==========================================================
# STAGE 5: META DATASET CONSTRUCTION
# XGB + ANN ONLY
# ==========================================================

import numpy as np
import pandas as pd
import joblib

# ==========================================================
# LOAD OOF PREDICTIONS
# ==========================================================

xgb_oof_train = joblib.load(
    "xgb_oof_train.pkl"
)

ann_oof_train = joblib.load(
    "ann_oof_train.pkl"
)

# ==========================================================
# LOAD TEST PREDICTIONS
# ==========================================================

xgb_test_pred = joblib.load(
    "xgb_test_pred.pkl"
)

ann_test_pred = joblib.load(
    "ann_test_pred.pkl"
)

# ==========================================================
# META TRAIN DATASET
# ==========================================================

X_meta_train = np.column_stack([
    xgb_oof_train,
    ann_oof_train
])

# ==========================================================
# META TEST DATASET
# ==========================================================

X_meta_test = np.column_stack([
    xgb_test_pred,
    ann_test_pred
])

# ==========================================================
# TARGET
# ==========================================================

y_meta_train = y_train
y_meta_test = y_test

# ==========================================================
# SAVE META DATASETS
# ==========================================================

joblib.dump(
    X_meta_train,
    "X_meta_train_XGB_ANN.pkl"
)

joblib.dump(
    X_meta_test,
    "X_meta_test_XGB_ANN.pkl"
)

joblib.dump(
    y_meta_train,
    "y_meta_train_XGB_ANN.pkl"
)

joblib.dump(
    y_meta_test,
    "y_meta_test_XGB_ANN.pkl"
)

# ==========================================================
# SAVE CSV FILES
# ==========================================================

pd.DataFrame(
    X_meta_train,
    columns=[
        "XGB",
        "ANN"
    ]
).to_csv(
    "meta_train_dataset_XGB_ANN.csv",
    index=False
)

pd.DataFrame(
    X_meta_test,
    columns=[
        "XGB",
        "ANN"
    ]
).to_csv(
    "meta_test_dataset_XGB_ANN.csv",
    index=False
)

# ==========================================================
# DISPLAY
# ==========================================================

print("\nMeta Dataset Construction Completed")

print(
    "X_meta_train shape:",
    X_meta_train.shape
)

print(
    "X_meta_test shape:",
    X_meta_test.shape
)

print(
    "Features Used:",
    ["XGB", "ANN"]
)


Meta Dataset Construction Completed
X_meta_train shape: (824, 2)
X_meta_test shape: (212, 2)
Features Used: ['XGB', 'ANN']


In [36]:
import numpy as np
import pandas as pd
import joblib

from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# ==========================================================
# LOAD META DATA
# ==========================================================

X_meta_train = joblib.load("X_meta_train_XGB_ANN.pkl")
X_meta_test = joblib.load("X_meta_test_XGB_ANN.pkl")

y_meta_train = joblib.load("y_meta_train_XGB_ANN.pkl")
y_meta_test = joblib.load("y_meta_test_XGB_ANN.pkl")

print("Train Shape:", X_meta_train.shape)
print("Test Shape :", X_meta_test.shape)

# ==========================================================
# METRICS
# ==========================================================

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def rrmse(y_true, y_pred):
    return rmse(y_true, y_pred) / np.mean(y_true)

def mape(y_true, y_pred):
    return np.mean(
        np.abs((y_true - y_pred) / y_true)
    ) * 100

# ==========================================================
# META XGB MODEL
# ==========================================================

meta_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# ==========================================================
# PARAMETER GRID
# ==========================================================

param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.03, 0.05, 0.10],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

# ==========================================================
# GRID SEARCH
# ==========================================================

grid = GridSearchCV(
    estimator=meta_xgb,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid.fit(
    X_meta_train,
    y_meta_train
)

# ==========================================================
# BEST MODEL
# ==========================================================

best_meta_xgb = grid.best_estimator_

print("\nBest Parameters:")
print(grid.best_params_)

print("\nBest CV R2:")
print(grid.best_score_)

# ==========================================================
# TEST PREDICTION
# ==========================================================

y_pred = best_meta_xgb.predict(
    X_meta_test
)

# ==========================================================
# TEST METRICS
# ==========================================================

test_r2 = r2_score(
    y_meta_test,
    y_pred
)

test_rmse = rmse(
    y_meta_test,
    y_pred
)

test_mae = mean_absolute_error(
    y_meta_test,
    y_pred
)

test_rrmse = rrmse(
    y_meta_test,
    y_pred
)

test_mape = mape(
    y_meta_test,
    y_pred
)

# ==========================================================
# RESULTS
# ==========================================================

print("\n===== META XGB (XGB + ANN) RESULTS =====")

print(f"CV R2    : {grid.best_score_:.6f}")
print(f"Test R2  : {test_r2:.6f}")
print(f"RMSE     : {test_rmse:.6f}")
print(f"MAE      : {test_mae:.6f}")
print(f"RRMSE    : {test_rrmse:.6f}")
print(f"MAPE (%) : {test_mape:.6f}")

# ==========================================================
# SAVE MODEL
# ==========================================================

joblib.dump(
    best_meta_xgb,
    "XGB_meta_XGB_ANN_model.pkl"
)

# ==========================================================
# SAVE PREDICTIONS
# ==========================================================

pred_df = pd.DataFrame({
    "Actual": y_meta_test,
    "Predicted": y_pred
})

pred_df.to_csv(
    "XGB_meta_XGB_ANN_predictions.csv",
    index=False
)

# ==========================================================
# SAVE RESULTS
# ==========================================================

results_df = pd.DataFrame([{
    "CV_R2": grid.best_score_,
    "Test_R2": test_r2,
    "RMSE": test_rmse,
    "MAE": test_mae,
    "RRMSE": test_rrmse,
    "MAPE": test_mape
}])

results_df.to_csv(
    "XGB_meta_XGB_ANN_results.csv",
    index=False
)

print("\nModel Saved -> XGB_meta_XGB_ANN_model.pkl")
print("Predictions Saved -> XGB_meta_XGB_ANN_predictions.csv")
print("Results Saved -> XGB_meta_XGB_ANN_results.csv")

Train Shape: (824, 2)
Test Shape : (212, 2)
Fitting 5 folds for each of 108 candidates, totalling 540 fits

Best Parameters:
{'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 200, 'subsample': 1.0}

Best CV R2:
0.997214017267137

===== META XGB (XGB + ANN) RESULTS =====
CV R2    : 0.997214
Test R2  : 0.996967
RMSE     : 0.874572
MAE      : 0.515991
RRMSE    : 0.003116
MAPE (%) : 0.192286

Model Saved -> XGB_meta_XGB_ANN_model.pkl
Predictions Saved -> XGB_meta_XGB_ANN_predictions.csv
Results Saved -> XGB_meta_XGB_ANN_results.csv


# 9- MODEL WITHOUT XG

In [37]:
# ==========================================================
# STAGE 5: META DATASET CONSTRUCTION
# RF + ANN ONLY
# ==========================================================

import numpy as np
import pandas as pd
import joblib

# ==========================================================
# LOAD OOF PREDICTIONS
# ==========================================================

rf_oof_train = joblib.load(
    "rf_oof_train.pkl"
)

ann_oof_train = joblib.load(
    "ann_oof_train.pkl"
)

# ==========================================================
# LOAD TEST PREDICTIONS
# ==========================================================

rf_test_pred = joblib.load(
    "rf_test_pred.pkl"
)

ann_test_pred = joblib.load(
    "ann_test_pred.pkl"
)

# ==========================================================
# META TRAIN DATASET
# ==========================================================

X_meta_train = np.column_stack([
    rf_oof_train,
    ann_oof_train
])

# ==========================================================
# META TEST DATASET
# ==========================================================

X_meta_test = np.column_stack([
    rf_test_pred,
    ann_test_pred
])

# ==========================================================
# TARGET
# ==========================================================

y_meta_train = y_train
y_meta_test = y_test

# ==========================================================
# SAVE META DATASETS
# ==========================================================

joblib.dump(
    X_meta_train,
    "X_meta_train_RF_ANN.pkl"
)

joblib.dump(
    X_meta_test,
    "X_meta_test_RF_ANN.pkl"
)

joblib.dump(
    y_meta_train,
    "y_meta_train_RF_ANN.pkl"
)

joblib.dump(
    y_meta_test,
    "y_meta_test_RF_ANN.pkl"
)

# ==========================================================
# SAVE CSV FILES
# ==========================================================

pd.DataFrame(
    X_meta_train,
    columns=[
        "RF",
        "ANN"
    ]
).to_csv(
    "meta_train_dataset_RF_ANN.csv",
    index=False
)

pd.DataFrame(
    X_meta_test,
    columns=[
        "RF",
        "ANN"
    ]
).to_csv(
    "meta_test_dataset_RF_ANN.csv",
    index=False
)

# ==========================================================
# DISPLAY
# ==========================================================

print("\nMeta Dataset Construction Completed")

print(
    "X_meta_train shape:",
    X_meta_train.shape
)

print(
    "X_meta_test shape:",
    X_meta_test.shape
)

print(
    "Features Used:",
    ["RF", "ANN"]
)


Meta Dataset Construction Completed
X_meta_train shape: (824, 2)
X_meta_test shape: (212, 2)
Features Used: ['RF', 'ANN']


In [38]:
import numpy as np
import pandas as pd
import joblib

from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# ==========================================================
# LOAD META DATA (RF + ANN)
# ==========================================================

X_meta_train = joblib.load(
    "X_meta_train_RF_ANN.pkl"
)

X_meta_test = joblib.load(
    "X_meta_test_RF_ANN.pkl"
)

y_meta_train = joblib.load(
    "y_meta_train_RF_ANN.pkl"
)

y_meta_test = joblib.load(
    "y_meta_test_RF_ANN.pkl"
)

print("Train Shape:", X_meta_train.shape)
print("Test Shape :", X_meta_test.shape)

# ==========================================================
# METRICS
# ==========================================================

def rmse(y_true, y_pred):
    return np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

def rrmse(y_true, y_pred):
    return rmse(
        y_true,
        y_pred
    ) / np.mean(y_true)

def mape(y_true, y_pred):
    return np.mean(
        np.abs(
            (y_true - y_pred) / y_true
        )
    ) * 100

# ==========================================================
# META XGB MODEL
# ==========================================================

meta_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# ==========================================================
# HYPERPARAMETER GRID
# ==========================================================

param_grid = {

    "n_estimators": [
        100,
        200,
        300
    ],

    "learning_rate": [
        0.03,
        0.05,
        0.10
    ],

    "max_depth": [
        2,
        3,
        4
    ],

    "subsample": [
        0.8,
        1.0
    ],

    "colsample_bytree": [
        0.8,
        1.0
    ]
}

# ==========================================================
# GRID SEARCH
# ==========================================================

grid = GridSearchCV(
    estimator=meta_xgb,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid.fit(
    X_meta_train,
    y_meta_train
)

# ==========================================================
# BEST MODEL
# ==========================================================

best_meta_xgb = grid.best_estimator_

print("\nBest Parameters:")
print(grid.best_params_)

print("\nBest CV R2:")
print(grid.best_score_)

# ==========================================================
# TEST PREDICTION
# ==========================================================

y_pred = best_meta_xgb.predict(
    X_meta_test
)

# ==========================================================
# TEST METRICS
# ==========================================================

test_r2 = r2_score(
    y_meta_test,
    y_pred
)

test_rmse = rmse(
    y_meta_test,
    y_pred
)

test_mae = mean_absolute_error(
    y_meta_test,
    y_pred
)

test_rrmse = rrmse(
    y_meta_test,
    y_pred
)

test_mape = mape(
    y_meta_test,
    y_pred
)

# ==========================================================
# RESULTS
# ==========================================================

print("\n===== META XGB (RF + ANN) RESULTS =====")

print(f"CV R2    : {grid.best_score_:.6f}")
print(f"Test R2  : {test_r2:.6f}")
print(f"RMSE     : {test_rmse:.6f}")
print(f"MAE      : {test_mae:.6f}")
print(f"RRMSE    : {test_rrmse:.6f}")
print(f"MAPE (%) : {test_mape:.6f}")

# ==========================================================
# SAVE MODEL
# ==========================================================

joblib.dump(
    best_meta_xgb,
    "XGB_meta_RF_ANN_model.pkl"
)

# ==========================================================
# SAVE PREDICTIONS
# ==========================================================

pred_df = pd.DataFrame({
    "Actual": y_meta_test,
    "Predicted": y_pred
})

pred_df.to_csv(
    "XGB_meta_RF_ANN_predictions.csv",
    index=False
)

# ==========================================================
# SAVE RESULTS
# ==========================================================

results_df = pd.DataFrame([{
    "CV_R2": grid.best_score_,
    "Test_R2": test_r2,
    "RMSE": test_rmse,
    "MAE": test_mae,
    "RRMSE": test_rrmse,
    "MAPE": test_mape
}])

results_df.to_csv(
    "XGB_meta_RF_ANN_results.csv",
    index=False
)

# ==========================================================
# FINISH
# ==========================================================

print(
    "\nModel Saved -> XGB_meta_RF_ANN_model.pkl"
)

print(
    "Predictions Saved -> XGB_meta_RF_ANN_predictions.csv"
)

print(
    "Results Saved -> XGB_meta_RF_ANN_results.csv"
)

Train Shape: (824, 2)
Test Shape : (212, 2)
Fitting 5 folds for each of 108 candidates, totalling 540 fits

Best Parameters:
{'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}

Best CV R2:
0.994220439441538

===== META XGB (RF + ANN) RESULTS =====
CV R2    : 0.994220
Test R2  : 0.998165
RMSE     : 0.680203
MAE      : 0.380944
RRMSE    : 0.002424
MAPE (%) : 0.143856

Model Saved -> XGB_meta_RF_ANN_model.pkl
Predictions Saved -> XGB_meta_RF_ANN_predictions.csv
Results Saved -> XGB_meta_RF_ANN_results.csv


# LAVAN PREDICTION 

In [54]:
# =====================================================
# STACKING PREDICTION + METRICS
# COMPOSITION MODELS ONLY
# RF + XGB + ANN -> META XGB
# =====================================================

import pandas as pd
import numpy as np
import joblib

from tensorflow.keras.models import load_model

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# =====================================================
# LOAD MODELS
# =====================================================

rf_model = joblib.load(
    "RF_BaseLearner1.pkl"
)

xgb_model = joblib.load(
    "XGB_BaseLearner1.pkl"
)

ann_model = load_model(
    "ann_base_final.keras"
)

X_scaler = joblib.load(
    "ann_X_scaler.pkl"
)

y_scaler = joblib.load(
    "ann_y_scaler.pkl"
)

meta_model = joblib.load(
    "XGB_Meta_Model.pkl"
)

# =====================================================
# METRICS
# =====================================================

def rmse(y_true, y_pred):
    return np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

def rrmse(y_true, y_pred):
    return (
        rmse(y_true, y_pred)
        /
        np.mean(y_true)
    ) * 100

def mape(y_true, y_pred):
    return np.mean(
        np.abs(
            (y_true - y_pred)
            /
            y_true
        )
    ) * 100

# =====================================================
# PRESSURES (psia)
# =====================================================

pressures = [

    157,
    210,
    263,
    421,
    525,
    631,
    841,
    1157,
    1526,
    2263,
    3260

]

# =====================================================
# EXPERIMENTAL DATA
# =====================================================

experimental = np.array([

    275.09,
    277.03,
    279.04,
    281.04,
    283.12,
    285.14,
    287.08,
    289.08,
    290.96,
    293.04,
    295.04

])

# =====================================================
# HYSYS RESULTS
# =====================================================

hysys = np.array([

    273.78,
    276.21,
    278.10,
    282.00,
    283.79,
    285.20,
    287.29,
    289.38,
    290.98,
    293.30,
    295.87

])

# =====================================================
# COMPOSITION INPUT
# =====================================================

X_full = pd.DataFrame({
    "Water":
        [0.0] * len(pressures),

    "Methane":
        [89.19] * len(pressures),

    "Ethane":
        [4.24] * len(pressures),

    "Propane":
        [1.14] * len(pressures),

    "Isobutane":
        [0.28] * len(pressures),

    "Butane":
        [0.30] * len(pressures),

    "Isopentane":
        [0.15] * len(pressures),

    "Pentane":
        [0.10] * len(pressures),

    "Hexane":
        [0.09] * len(pressures),

    "Heptane":
        [0.07] * len(pressures),

    "Carbon Dioxide":
        [4.44] * len(pressures),

    "Pressure (psia)":
        pressures

})

# =====================================================
# RF PREDICTION
# =====================================================

rf_pred = rf_model.predict(
    X_full
)

# =====================================================
# XGB PREDICTION
# =====================================================

xgb_pred = xgb_model.predict(
    X_full
)

# =====================================================
# ANN PREDICTION
# =====================================================

X_scaled = X_scaler.transform(
    X_full
)

ann_scaled_pred = ann_model.predict(
    X_scaled,
    verbose=0
)

ann_pred = y_scaler.inverse_transform(
    ann_scaled_pred
).flatten()

# =====================================================
# META FEATURES
# =====================================================

X_meta = np.column_stack([

    rf_pred,
    xgb_pred,
    ann_pred

])

# =====================================================
# STACKING PREDICTION
# =====================================================

stack_pred = meta_model.predict(
    X_meta
)

# =====================================================
# PREDICTION TABLE
# =====================================================

predictions = pd.DataFrame({

    "Pressure (psia)":
        pressures,

    "Experimental":
        experimental,

    "HYSYS":
        np.round(
            hysys,
            3
        ),

    "RF_Base":
        np.round(
            rf_pred,
            3
        ),

    "XGB_Base":
        np.round(
            xgb_pred,
            3
        ),

    "ANN_Base":
        np.round(
            ann_pred,
            3
        ),

    "Stacking":
        np.round(
            stack_pred,
            3
        )

})

print("\n")
print("="*100)
print("PREDICTIONS")
print("="*100)

print(predictions)

# =====================================================
# METRICS
# =====================================================

models = {

    "HYSYS":
        hysys,

    "RF_Base":
        rf_pred,

    "XGB_Base":
        xgb_pred,

    "ANN_Base":
        ann_pred,

    "Stacking":
        stack_pred

}

metrics = []

for name, pred in models.items():

    metrics.append({

        "Model":
            name,

        "R2":
            r2_score(
                experimental,
                pred
            ),

        "RMSE":
            rmse(
                experimental,
                pred
            ),

        "MAE":
            mean_absolute_error(
                experimental,
                pred
            ),

        "RRMSE (%)":
            rrmse(
                experimental,
                pred
            ),

        "MAPE (%)":
            mape(
                experimental,
                pred
            )

    })

# =====================================================
# RESULTS TABLE
# =====================================================

metrics_df = pd.DataFrame(
    metrics
)

metrics_df = metrics_df.sort_values(
    by="RMSE"
)

print("\n")
print("="*100)
print("MODEL COMPARISON")
print("="*100)

print(
    metrics_df.round(6)
)

# =====================================================
# SAVE RESULTS
# =====================================================

predictions.to_csv(
    "Lavan3_Predictions.csv",
    index=False
)

metrics_df.to_csv(
    "Lavan3_Metrics.csv",
    index=False
)

print("\nFiles Saved:")
print("Lavan3_Predictions.csv")
print("Lavan3_Metrics.csv")

C:\Users\lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:432: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(




PREDICTIONS
    Pressure (psia)  Experimental   HYSYS  RF_Base    XGB_Base    ANN_Base  \
0               157        275.09  273.78  273.542  273.562988  272.944000   
1               210        277.03  276.21  276.495  276.445007  275.191986   
2               263        279.04  278.10  278.167  278.256012  277.212006   
3               421        281.04  282.00  282.126  282.074005  281.665009   
4               525        283.12  283.79  283.972  284.247009  283.579987   
5               631        285.14  285.20  285.399  285.582001  285.088989   
6               841        287.08  287.29  287.333  287.391998  287.253998   
7              1157        289.08  289.38  289.447  289.833008  289.355988   
8              1526        290.96  290.98  291.087  291.127991  291.002014   
9              2263        293.04  293.30  293.046  293.382996  293.386993   
10             3260        295.04  295.87  294.203  294.170013  295.997009   

      Stacking  
0   273.959991  
1   276.010010 

In [55]:
# =====================================================
# STACKING PREDICTION + METRICS
# COMPOSITION MODELS ONLY
# RF + XGB + ANN
# XGB META + GPR META
# =====================================================

import pandas as pd
import numpy as np
import joblib

from tensorflow.keras.models import load_model

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# =====================================================
# LOAD BASE MODELS
# =====================================================

rf_model = joblib.load(
    "RF_BaseLearner1.pkl"
)

xgb_model = joblib.load(
    "XGB_BaseLearner1.pkl"
)

ann_model = load_model(
    "ann_base_final.keras"
)

X_scaler = joblib.load(
    "ann_X_scaler.pkl"
)

y_scaler = joblib.load(
    "ann_y_scaler.pkl"
)

# =====================================================
# LOAD META MODELS
# =====================================================

xgb_meta = joblib.load(
    "XGB_Meta_Model.pkl"
)

gpr_meta = joblib.load(
    "GPR_meta_model.pkl"
)

# =====================================================
# METRICS
# =====================================================

def rmse(y_true, y_pred):
    return np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

def rrmse(y_true, y_pred):
    return (
        rmse(y_true, y_pred)
        /
        np.mean(y_true)
    ) * 100

def mape(y_true, y_pred):
    return np.mean(
        np.abs(
            (y_true - y_pred)
            /
            y_true
        )
    ) * 100

# =====================================================
# PRESSURES (psia)
# =====================================================

pressures = [

    157,
    210,
    263,
    421,
    525,
    631,
    841,
    1157,
    1526,
    2263,
    3260

]

# =====================================================
# EXPERIMENTAL DATA
# =====================================================

experimental = np.array([

    275.09,
    277.03,
    279.04,
    281.04,
    283.12,
    285.14,
    287.08,
    289.08,
    290.96,
    293.04,
    295.04

])

# =====================================================
# HYSYS RESULTS
# =====================================================

hysys = np.array([

    273.78,
    276.21,
    278.10,
    282.00,
    283.79,
    285.20,
    287.29,
    289.38,
    290.98,
    293.30,
    295.87

])

# =====================================================
# COMPOSITION INPUT
# MUST MATCH TRAINING FEATURES
# =====================================================

X_full = pd.DataFrame({

    "Water":
        [0.0] * len(pressures),

    "Methane":
        [89.19] * len(pressures),

    "Ethane":
        [4.24] * len(pressures),

    "Propane":
        [1.14] * len(pressures),

    "Isobutane":
        [0.28] * len(pressures),

    "Butane":
        [0.30] * len(pressures),

    "Isopentane":
        [0.15] * len(pressures),

    "Pentane":
        [0.10] * len(pressures),

    "Hexane":
        [0.09] * len(pressures),

    "Heptane":
        [0.07] * len(pressures),

    "Carbon Dioxide":
        [4.44] * len(pressures),

    "Pressure (psia)":
        pressures

})

print("\nFeatures Used:")
print(X_full.columns.tolist())
print("Shape:", X_full.shape)

# =====================================================
# RF PREDICTION
# =====================================================

rf_pred = rf_model.predict(
    X_full
)

# =====================================================
# XGB PREDICTION
# =====================================================

xgb_pred = xgb_model.predict(
    X_full
)

# =====================================================
# ANN PREDICTION
# =====================================================

X_scaled = X_scaler.transform(
    X_full
)

ann_scaled_pred = ann_model.predict(
    X_scaled,
    verbose=0
)

ann_pred = y_scaler.inverse_transform(
    ann_scaled_pred
).ravel()

# =====================================================
# META FEATURES
# =====================================================

X_meta = np.column_stack([

    rf_pred,
    xgb_pred,
    ann_pred

])

# =====================================================
# META PREDICTIONS
# =====================================================

xgb_stack_pred = xgb_meta.predict(
    X_meta
)

gpr_stack_pred = gpr_meta.predict(
    X_meta
)

# =====================================================
# PREDICTION TABLE
# =====================================================

predictions = pd.DataFrame({

    "Pressure (psia)":
        pressures,

    "Experimental":
        experimental,

    "HYSYS":
        np.round(
            hysys,
            3
        ),

    "RF_Base":
        np.round(
            rf_pred,
            3
        ),

    "XGB_Base":
        np.round(
            xgb_pred,
            3
        ),

    "ANN_Base":
        np.round(
            ann_pred,
            3
        ),

    "XGB_Stacking":
        np.round(
            xgb_stack_pred,
            3
        ),

    "GPR_Stacking":
        np.round(
            gpr_stack_pred,
            3
        )

})

print("\n")
print("="*120)
print("PREDICTIONS")
print("="*120)

print(predictions)

# =====================================================
# METRICS COMPARISON
# =====================================================

models = {

    "HYSYS":
        hysys,

    "RF_Base":
        rf_pred,

    "XGB_Base":
        xgb_pred,

    "ANN_Base":
        ann_pred,

    "XGB_Stacking":
        xgb_stack_pred,

    "GPR_Stacking":
        gpr_stack_pred

}

metrics = []

for name, pred in models.items():

    metrics.append({

        "Model":
            name,

        "R2":
            r2_score(
                experimental,
                pred
            ),

        "RMSE":
            rmse(
                experimental,
                pred
            ),

        "MAE":
            mean_absolute_error(
                experimental,
                pred
            ),

        "RRMSE (%)":
            rrmse(
                experimental,
                pred
            ),

        "MAPE (%)":
            mape(
                experimental,
                pred
            )

    })

metrics_df = pd.DataFrame(
    metrics
)

metrics_df = metrics_df.sort_values(
    by="RMSE"
)

print("\n")
print("="*120)
print("MODEL COMPARISON")
print("="*120)

print(
    metrics_df.round(6)
)

# =====================================================
# SAVE RESULTS
# =====================================================

predictions.to_csv(
    "Lavan3_All_Predictions.csv",
    index=False
)

metrics_df.to_csv(
    "Lavan3_All_Metrics.csv",
    index=False
)

print("\nFiles Saved:")
print("Lavan3_All_Predictions.csv")
print("Lavan3_All_Metrics.csv")


Features Used:
['Water', 'Methane', 'Ethane', 'Propane', 'Isobutane', 'Butane', 'Isopentane', 'Pentane', 'Hexane', 'Heptane', 'Carbon Dioxide', 'Pressure (psia)']
Shape: (11, 12)


C:\Users\lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:432: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(




PREDICTIONS
    Pressure (psia)  Experimental   HYSYS  RF_Base    XGB_Base    ANN_Base  \
0               157        275.09  273.78  273.542  273.562988  272.944000   
1               210        277.03  276.21  276.495  276.445007  275.191986   
2               263        279.04  278.10  278.167  278.256012  277.212006   
3               421        281.04  282.00  282.126  282.074005  281.665009   
4               525        283.12  283.79  283.972  284.247009  283.579987   
5               631        285.14  285.20  285.399  285.582001  285.088989   
6               841        287.08  287.29  287.333  287.391998  287.253998   
7              1157        289.08  289.38  289.447  289.833008  289.355988   
8              1526        290.96  290.98  291.087  291.127991  291.002014   
9              2263        293.04  293.30  293.046  293.382996  293.386993   
10             3260        295.04  295.87  294.203  294.170013  295.997009   

    XGB_Stacking  GPR_Stacking  
0     273.959991

In [56]:
# =====================================================
# STACKING PREDICTION FOR SALMAN DATASET
# RF + XGB + ANN + META XGB + META GPR
# =====================================================

import pandas as pd
import numpy as np
import joblib

from tensorflow.keras.models import load_model

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# =====================================================
# LOAD MODELS
# =====================================================

rf_model = joblib.load(
    "RF_BaseLearner1.pkl"
)

xgb_model = joblib.load(
    "XGB_BaseLearner1.pkl"
)

ann_model = load_model(
    "ann_base_final.keras"
)

X_scaler = joblib.load(
    "ann_X_scaler.pkl"
)

y_scaler = joblib.load(
    "ann_y_scaler.pkl"
)

meta_xgb = joblib.load(
    "XGB_Meta_Model.pkl"
)

meta_gpr = joblib.load(
    "GPR_meta_model.pkl"
)

# =====================================================
# METRICS FUNCTIONS
# =====================================================

def rmse(y_true, y_pred):

    return np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

def rrmse(y_true, y_pred):

    return (
        rmse(y_true, y_pred)
        /
        np.mean(y_true)
    ) * 100

def mape(y_true, y_pred):

    return np.mean(
        np.abs(
            (y_true - y_pred)
            /
            y_true
        )
    ) * 100

# =====================================================
# PRESSURES (psia)
# =====================================================

pressures = [

    150,
    250,
    350,
    500,
    700,
    900,
    1200,
    1600

]

# =====================================================
# EXPERIMENTAL DATA (K)
# =====================================================

experimental = np.array([

    274.8,
    277.6,
    280.4,
    283.2,
    286.0,
    288.7,
    291.5,
    293.2

])

# =====================================================
# XGBOOST PAPER VALUES
# =====================================================

xgboost_paper = np.array([

    275.35,
    280.25,
    282.61,
    285.73,
    287.95,
    289.48,
    291.22,
    292.62

])

# =====================================================
# SALMAN GAS COMPOSITION
# =====================================================

CH4  = 77.61
CO2  = 6.14
C2H6 = 8.81
C3H8 = 3.37

iC4  = 0.75
nC4  = 1.33

iC5  = 0.44
nC5  = 0.69

C6p  = 0.86

# =====================================================
# INPUT DATASET
# IMPORTANT:
# RF/XGB TRAINED WITH 12 FEATURES
# WATER COLUMN INCLUDED
# =====================================================

X_full = pd.DataFrame({

    "Water":
        [0.0] * len(pressures),

    "Methane":
        [CH4] * len(pressures),

    "Ethane":
        [C2H6] * len(pressures),

    "Propane":
        [C3H8] * len(pressures),

    "Isobutane":
        [iC4] * len(pressures),

    "Butane":
        [nC4] * len(pressures),

    "Isopentane":
        [iC5] * len(pressures),

    "Pentane":
        [nC5] * len(pressures),

    "Hexane":
        [C6p] * len(pressures),

    "Heptane":
        [0.0] * len(pressures),

    "Carbon Dioxide":
        [CO2] * len(pressures),

    "Pressure (psia)":
        pressures

})

# =====================================================
# CHECK FEATURES
# =====================================================

print("\n")
print("="*60)
print("INPUT FEATURES")
print("="*60)

print(X_full.columns.tolist())

print("\nNumber Features = ",
      X_full.shape[1])

# =====================================================
# RF PREDICTION
# =====================================================

rf_pred = rf_model.predict(
    X_full
)

# =====================================================
# XGB PREDICTION
# =====================================================

xgb_pred = xgb_model.predict(
    X_full
)

# =====================================================
# ANN PREDICTION
# =====================================================

X_scaled = X_scaler.transform(
    X_full
)

ann_scaled = ann_model.predict(
    X_scaled,
    verbose=0
)

ann_pred = y_scaler.inverse_transform(
    ann_scaled
).flatten()

# =====================================================
# META FEATURES
# =====================================================

X_meta = np.column_stack([

    rf_pred,
    xgb_pred,
    ann_pred

])

# =====================================================
# META XGB
# =====================================================

stack_xgb = meta_xgb.predict(
    X_meta
)

# =====================================================
# META GPR
# =====================================================

stack_gpr = meta_gpr.predict(
    X_meta
)

# =====================================================
# PREDICTIONS TABLE
# =====================================================

predictions = pd.DataFrame({

    "Pressure (psia)":
        pressures,

    "Experimental":
        experimental,

    "Paper_XGBoost":
        np.round(
            xgboost_paper,
            3
        ),

    "RF_Base":
        np.round(
            rf_pred,
            3
        ),

    "XGB_Base":
        np.round(
            xgb_pred,
            3
        ),

    "ANN_Base":
        np.round(
            ann_pred,
            3
        ),

    "Meta_XGB":
        np.round(
            stack_xgb,
            3
        ),

    "Meta_GPR":
        np.round(
            stack_gpr,
            3
        )

})

print("\n")
print("="*120)
print("PREDICTIONS")
print("="*120)

print(predictions)

# =====================================================
# METRICS
# =====================================================

models = {

    "Paper_XGBoost":
        xgboost_paper,

    "RF_Base":
        rf_pred,

    "XGB_Base":
        xgb_pred,

    "ANN_Base":
        ann_pred,

    "Meta_XGB":
        stack_xgb,

    "Meta_GPR":
        stack_gpr

}

metrics = []

for name, pred in models.items():

    metrics.append({

        "Model":
            name,

        "R2":
            r2_score(
                experimental,
                pred
            ),

        "RMSE":
            rmse(
                experimental,
                pred
            ),

        "MAE":
            mean_absolute_error(
                experimental,
                pred
            ),

        "RRMSE (%)":
            rrmse(
                experimental,
                pred
            ),

        "MAPE (%)":
            mape(
                experimental,
                pred
            )

    })

# =====================================================
# RESULTS TABLE
# =====================================================

metrics_df = pd.DataFrame(
    metrics
)

metrics_df = metrics_df.sort_values(
    by="RMSE"
)

print("\n")
print("="*120)
print("MODEL COMPARISON")
print("="*120)

print(
    metrics_df.round(6)
)

# =====================================================
# SAVE RESULTS
# =====================================================

predictions.to_csv(
    "Salman_Predictions.csv",
    index=False
)

metrics_df.to_csv(
    "Salman_Metrics.csv",
    index=False
)

print("\nFiles Saved:")
print("Salman_Predictions.csv")
print("Salman_Metrics.csv")



INPUT FEATURES
['Water', 'Methane', 'Ethane', 'Propane', 'Isobutane', 'Butane', 'Isopentane', 'Pentane', 'Hexane', 'Heptane', 'Carbon Dioxide', 'Pressure (psia)']

Number Features =  12


C:\Users\lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:432: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\lenovo\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:432: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(




PREDICTIONS
   Pressure (psia)  Experimental  Paper_XGBoost  RF_Base    XGB_Base  \
0              150         274.8         275.35  275.650  276.510010   
1              250         277.6         280.25  280.174  280.813995   
2              350         280.4         282.61  282.899  283.824005   
3              500         283.2         285.73  285.765  286.115997   
4              700         286.0         287.95  288.133  288.539001   
5              900         288.7         289.48  289.806  289.904999   
6             1200         291.5         291.22  291.285  291.453003   
7             1600         293.2         292.62  292.131  293.007996   

     ANN_Base    Meta_XGB  Meta_GPR  
0  277.817993  276.648010   277.029  
1  280.852997  280.507996   280.760  
2  283.398010  283.661987   283.772  
3  286.269989  285.903992   286.199  
4  288.846985  288.235992   288.552  
5  290.609985  289.890991   289.883  
6  292.419006  291.464996   291.507  
7  294.079987  292.635986   292.9